# JetBot Tape Line Follower
Run cells top to bottom. Switch between **Canny Edge** and **Adaptive Threshold** detection modes.
Tune sliders while watching the live mask. Toggle **Enable Drive** when detection looks solid.

In [1]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import json
from jetbot import Robot, Camera

In [2]:
camera = Camera.instance(width=300, height=300, fps=10)
robot = Robot()

In [3]:
# ── All-in-one dashboard ──

# Camera feeds
cam_img  = widgets.Image(format='jpeg', width=300, height=300)
mask_img = widgets.Image(format='jpeg', width=300, height=300)
info_lbl = widgets.HTML(value='<i>Waiting for frames...</i>')

# ── Detection mode ──
mode = widgets.Dropdown(
    options=['Canny Edge', 'Adaptive Threshold'],
    value='Canny Edge', description='Mode')

# ── Canny sliders ──
blur_k     = widgets.IntSlider(value=5,   min=1, max=21, step=2, description='Blur K')
canny_low  = widgets.IntSlider(value=50,  min=0, max=255, description='Canny Low')
canny_high = widgets.IntSlider(value=150, min=0, max=255, description='Canny High')
dilate_k   = widgets.IntSlider(value=3,   min=1, max=15, step=2, description='Dilate K')

# ── Adaptive threshold sliders ──
block_size = widgets.IntSlider(value=11,  min=3,  max=51, step=2, description='Block Size')
c_offset   = widgets.IntSlider(value=2,   min=-20, max=20, description='C Offset')
invert     = widgets.ToggleButton(value=False, description='Invert',
                button_style='', layout=widgets.Layout(width='100px'))

# ── Shared sliders ──
min_run_w  = widgets.IntSlider(value=3,   min=1, max=30, description='Min Run W')
roi_pct    = widgets.FloatSlider(value=0.4,  min=0.1, max=1.0, step=0.05, description='ROI Bot %')
scan_rows  = widgets.IntSlider(value=10,  min=3, max=30, description='Scan Rows')
kp         = widgets.FloatSlider(value=0.4,  min=0.0, max=2.0, step=0.01, description='Kp')
ki         = widgets.FloatSlider(value=0.0,  min=0.0, max=0.5, step=0.005, description='Ki')
kd         = widgets.FloatSlider(value=0.1,  min=0.0, max=1.0, step=0.01, description='Kd')
base_speed = widgets.FloatSlider(value=0.25, min=0.05, max=0.6, step=0.01, description='Base Speed')
curve_slow = widgets.FloatSlider(value=0.5,  min=0.0, max=1.0, step=0.05, description='Curve Slow')
lookahead  = widgets.FloatSlider(value=0.3,  min=0.0, max=1.0, step=0.05, description='Look-ahead')
min_wheel  = widgets.FloatSlider(value=0.0,  min=-0.2, max=0.15, step=0.01, description='Min Wheel')

# ── Drive toggle ──
drive_enabled = widgets.ToggleButton(
    value=False, description='Enable Drive',
    button_style='danger', icon='car',
    layout=widgets.Layout(width='160px', height='40px'))

def on_drive_toggle(change):
    if change['new']:
        drive_enabled.button_style = 'success'
        drive_enabled.description = 'Driving!'
    else:
        drive_enabled.button_style = 'danger'
        drive_enabled.description = 'Enable Drive'
        robot.stop()
drive_enabled.observe(on_drive_toggle, names='value')

# ── Show/hide mode-specific sliders ──
canny_box    = widgets.VBox([blur_k, canny_low, canny_high, dilate_k])
adaptive_box = widgets.VBox([block_size, c_offset, invert])
adaptive_box.layout.display = 'none'

def on_mode_change(change):
    if change['new'] == 'Canny Edge':
        canny_box.layout.display = ''
        adaptive_box.layout.display = 'none'
    else:
        canny_box.layout.display = 'none'
        adaptive_box.layout.display = ''
mode.observe(on_mode_change, names='value')

# ── Save / Load ──
save_btn = widgets.Button(description='Save', button_style='info', icon='save')
load_btn = widgets.Button(description='Load', button_style='warning', icon='folder-open')
status_lbl = widgets.Label(value='')
CFG = 'line_follower_config.json'

ALL_PARAMS = [('mode',mode),('blur_k',blur_k),('canny_low',canny_low),
              ('canny_high',canny_high),('dilate_k',dilate_k),
              ('block_size',block_size),('c_offset',c_offset),('invert',invert),
              ('min_run_w',min_run_w),('roi_pct',roi_pct),('scan_rows',scan_rows),
              ('kp',kp),('ki',ki),('kd',kd),('base_speed',base_speed),
              ('curve_slow',curve_slow),('lookahead',lookahead),('min_wheel',min_wheel)]

def save_config(_):
    with open(CFG,'w') as f:
        json.dump({k: w.value for k,w in ALL_PARAMS}, f, indent=2)
    status_lbl.value = 'Saved!'
def load_config(_):
    try:
        with open(CFG) as f:
            cfg = json.load(f)
        for k, w in ALL_PARAMS:
            if k in cfg: w.value = cfg[k]
        status_lbl.value = 'Loaded!'
    except FileNotFoundError:
        status_lbl.value = 'No config file.'
save_btn.on_click(save_config)
load_btn.on_click(load_config)

# ── Unified layout ──
left_col = widgets.VBox([
    widgets.HBox([cam_img, mask_img]),
    info_lbl
])

right_col = widgets.VBox([
    mode,
    widgets.HTML('<b>Detection</b>'),
    canny_box, adaptive_box, min_run_w,
    widgets.HTML('<hr style="margin:4px 0"><b>PID / Steering</b>'),
    roi_pct, scan_rows, kp, ki, kd,
    widgets.HTML('<hr style="margin:4px 0"><b>Speed / Drive</b>'),
    base_speed, curve_slow, lookahead, min_wheel,
    drive_enabled,
    widgets.HBox([save_btn, load_btn, status_lbl]),
], layout=widgets.Layout(padding='0 0 0 12px'))

display(widgets.HBox([left_col, right_col]))

# ── PID state ──
_integral = 0.0
_prev_err = 0.0

def get_runs(row):
    """Find contiguous runs of white pixels in a row.
    Returns list of (start, end) for each run."""
    padded = np.concatenate(([0], row, [0]))
    diff = np.diff(padded.astype(np.int16))
    starts = np.where(diff > 0)[0]
    ends   = np.where(diff < 0)[0]
    return list(zip(starts, ends))

def find_lane_center(mask):
    """Scan rows for two tape strips and find the lane center between them.
    Uses inner edges: right edge of left tape, left edge of right tape.
    Falls back to single-tape center if only one strip is visible.
    Returns list of (row, left_inner, right_inner, center_x)."""
    rh, rw = mask.shape
    n = scan_rows.value
    row_indices = np.linspace(rh - 1, 0, n, dtype=int)
    mrw = min_run_w.value
    points = []
    for r in row_indices:
        runs = get_runs(mask[r])
        # Filter out tiny noise runs
        runs = [(s, e) for s, e in runs if (e - s) >= mrw]
        if len(runs) >= 2:
            # Two tape strips found - use inner edges
            # Left tape = leftmost run, right tape = rightmost run
            left_tape  = runs[0]    # (start, end) of left strip
            right_tape = runs[-1]   # (start, end) of right strip
            li = int(left_tape[1] - 1)   # inner edge of left tape (right side)
            ri = int(right_tape[0])       # inner edge of right tape (left side)
            cx = (li + ri) // 2
            points.append((int(r), li, ri, cx))
        elif len(runs) == 1:
            # Only one tape visible - use its center
            s, e = runs[0]
            lx = int(s)
            rx = int(e - 1)
            cx = (lx + rx) // 2
            points.append((int(r), lx, rx, cx))
    return points

def process_frame(change):
    global _integral, _prev_err
    frame = change['new']
    h, w = frame.shape[:2]

    roi_top = int(h * (1.0 - roi_pct.value))
    roi = frame[roi_top:, :]
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

    bk = blur_k.value | 1
    blurred = cv2.GaussianBlur(gray, (bk, bk), 0)

    if mode.value == 'Canny Edge':
        edges = cv2.Canny(blurred, canny_low.value, canny_high.value)
        dk = dilate_k.value | 1
        mask = cv2.dilate(edges, np.ones((dk, dk), np.uint8), iterations=2)
    else:
        bs = block_size.value | 1
        if bs < 3: bs = 3
        thresh_type = cv2.THRESH_BINARY_INV if not invert.value else cv2.THRESH_BINARY
        mask = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     thresh_type, bs, c_offset.value)
        kern = np.ones((3, 3), np.uint8)
        mask = cv2.erode(mask, kern, iterations=1)
        mask = cv2.dilate(mask, kern, iterations=2)

    points = find_lane_center(mask)

    # Draw overlay
    ov = frame.copy()
    cv2.line(ov, (0, roi_top), (w, roi_top), (0, 0, 255), 2)
    cv2.line(ov, (w // 2, roi_top), (w // 2, h), (128, 128, 128), 1)

    if len(points) >= 2:
        for r, lx, rx, cx in points:
            yr = r + roi_top
            cv2.circle(ov, (lx, yr), 4, (0, 0, 255), -1)   # left inner edge
            cv2.circle(ov, (rx, yr), 4, (255, 0, 0), -1)    # right inner edge
            cv2.circle(ov, (cx, yr), 5, (0, 255, 0), -1)    # lane center

        for i in range(len(points) - 1):
            y1 = points[i][0] + roi_top
            y2 = points[i+1][0] + roi_top
            cv2.line(ov, (points[i][3], y1), (points[i+1][3], y2), (0, 255, 0), 2)

        # Split into near/far for look-ahead
        mid = len(points) // 2
        near_pts = points[:mid] if mid > 0 else points
        far_pts  = points[mid:]

        near_weights = np.linspace(2.0, 1.0, len(near_pts))
        near_centers = np.array([p[3] for p in near_pts])
        near_cx = np.average(near_centers, weights=near_weights)
        near_err = (near_cx - w / 2) / (w / 2)

        far_centers = np.array([p[3] for p in far_pts])
        far_cx = np.mean(far_centers)
        far_err = (far_cx - w / 2) / (w / 2)

        la = lookahead.value
        error = (1.0 - la) * near_err + la * far_err

        # PID
        _integral = _integral * 0.9 + error
        deriv = error - _prev_err
        steer = kp.value * error + ki.value * _integral + kd.value * deriv
        _prev_err = error
        steer = max(-1.0, min(1.0, steer))

        # Curve slowdown
        speed = base_speed.value * (1.0 - curve_slow.value * abs(steer))

        # Motor output with min wheel floor
        lm = speed + steer
        rm = speed - steer
        mw = min_wheel.value
        lm = max(mw, min(1.0, lm))
        rm = max(mw, min(1.0, rm))

        if drive_enabled.value:
            robot.set_motors(lm, rm)

        # Draw targets
        bot_y = points[0][0] + roi_top
        top_y = far_pts[0][0] + roi_top if far_pts else bot_y
        cv2.circle(ov, (int(near_cx), bot_y), 10, (0, 255, 255), -1)
        cv2.circle(ov, (int(far_cx), top_y), 8, (255, 255, 0), 2)
        blend_x = int((1.0 - la) * near_cx + la * far_cx)
        cv2.line(ov, (w // 2, bot_y), (blend_x, bot_y), (255, 0, 255), 2)

        avg_lane = np.mean([p[2] - p[1] for p in points])
        info_lbl.value = (f'<b style="color:green">TRACKING</b> &nbsp; '
                         f'err:{error:+.2f} &nbsp; steer:{steer:+.2f} &nbsp; '
                         f'spd:{speed:.2f} &nbsp; '
                         f'L:{lm:.2f} R:{rm:.2f} &nbsp; '
                         f'lane:{avg_lane:.0f}px &nbsp; rows:{len(points)}')
    else:
        _no_line(ov, h)

    _, j1 = cv2.imencode('.jpg', ov)
    cam_img.value = j1.tobytes()
    mask_full = np.zeros((h, w), np.uint8)
    mask_full[roi_top:, :] = mask
    _, j2 = cv2.imencode('.jpg', cv2.cvtColor(mask_full, cv2.COLOR_GRAY2BGR))
    mask_img.value = j2.tobytes()

def _no_line(ov, h):
    global _integral, _prev_err
    _integral = 0.0
    _prev_err = 0.0
    if drive_enabled.value:
        robot.stop()
    cv2.putText(ov, 'LINE LOST', (10, h // 2),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    info_lbl.value = '<b style="color:red">LINE LOST</b> - motors stopped'

camera.observe(process_frame, names='value')
print('Dashboard active. Adjust sliders live, toggle Enable Drive when ready.')

Dashboard active. Adjust sliders live, toggle Enable Drive when ready.


In [4]:
# ── STOP ── run this cell to halt everything
camera.unobserve_all()
robot.stop()
drive_enabled.value = False
print('Stopped.')

Stopped.
